In [1]:
import os
import gc
import sys
import glob
import numpy as np
import pandas as pd
import netCDF4 as nc
from datetime import datetime, timedelta
from matplotlib.cm import get_cmap
import matplotlib.pyplot as plt
from matplotlib import cm
from matplotlib import colors
import matplotlib.ticker as mticker
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import multiprocessing as mp

In [2]:
# To use PLUMBER2_GPP_common_utils, change directory to where it exists
os.chdir('/srv/ccrc/LandAP/z5218916/script/PLUMBER2/LSM_GPP_PLUMBER2')
from PLUMBER2_GPP_common_utils import *

In [3]:
# Path of PLUMBER 2 dataset
PLUMBER2_path      = "/srv/ccrc/LandAP/z5218916/data/PLUMBER2/"
PLUMBER2_flux_path = "/srv/ccrc/LandAP/z5218916/data/Fluxnet_data/Post-processed_PLUMBER2_outputs/Nc_files/Flux/"
PLUMBER2_met_path  = "/srv/ccrc/LandAP/z5218916/data/Fluxnet_data/Post-processed_PLUMBER2_outputs/Nc_files/Met/"

site_names, IGBP_types, clim_types, model_names = load_default_list()

remove_site        = get_removed_site_names()

models_calc_LAI   = ['ORC2_r6593','ORC2_r6593_CO2','ORC3_r7245_NEE','ORC3_r8120','GFDL','SDGVM','QUINCY','NoahMPv401']
model_LAI_names   = {'ORC2_r6593':'lai','ORC2_r6593_CO2':'lai','ORC3_r7245_NEE':'lai','ORC3_r8120':'lai',
                     'GFDL':'lai', 'SDGVM':'lai','QUINCY':'LAI','NoahMPv401':'LAI'} #

# Calculate remaining sites
set_site_names      = set(site_names)
set_remove_site     = set(remove_site)
remain_sites        = set_site_names - set_remove_site
remain_sites        = list(remain_sites)

In [18]:
model_colors ={0:'red', 1: 'darkorange',2:'orange',3:'gold',4:'yellowgreen',5:'green',6:'mediumseagreen',
               7:'lime',8:'aquamarine',9:'cyan',10:'dodgerblue',11:'blue',12:'darkolivegreen',
               13:'forestgreen',14:'lime',15:'gold', 16:'orange',17:'pink',18:'pink',19:'red',20:'deeppink',
               21:'mediumorchid',22: 'darkviolet',}

IGBP_colors  = set_IGBP_colors()
clim_colors  = set_clim_colors()

<h4 style="color:green;">Monthly mean NEE</h4>  

In [19]:
def save_monthly_mean(var_name, model_in, per_LAI=False):
    
    var       = np.zeros((12,170))
    Site_name = [""] * 170    # Creates a list with 170 empty strings
    lat       = np.zeros(170)
    lon       = np.zeros(170)
    
    for i, site_name in enumerate(remain_sites): 
        
        Site_name[i]       = site_name
        
        PLUMBER2_path_site = f"/srv/ccrc/LandAP/z5218916/script/PLUMBER2/LSM_GPP_PLUMBER2/nc_files/{site_name}.nc"
        PLUMBER2_met_path  = "/srv/ccrc/LandAP/z5218916/data/Fluxnet_data/Post-processed_PLUMBER2_outputs/Nc_files/Met/"
        PLUMBER2_flux_path = "/srv/ccrc/LandAP/z5218916/data/Fluxnet_data/Post-processed_PLUMBER2_outputs/Nc_files/Flux/"
        file_path          = glob.glob(PLUMBER2_flux_path+"/*"+site_name+"*.nc")
        
        with nc.Dataset(PLUMBER2_path_site, mode='r') as f:
            try:
                if var_name == 'NEE':
                    if model_in == 'NoahMPv401' or model_in == 'GFDL' or model_in == 'STEMMUS-SCOPE':
                        var_tmp = f.variables[model_in + '_NEE'][:].data*(-1)
                    else:
                        var_tmp = f.variables[model_in + '_NEE'][:].data
                elif var_name == 'GPP':
                    var_tmp = f.variables[model_in + '_GPP'][:].data

                # Read time
                time   = nc.num2date(f.variables['CABLE_time'][:],f.variables['CABLE_time'].units,
                                     only_use_cftime_datetimes=False,only_use_python_datetimes=True)
                ntime  = len(time)
                month  = np.zeros(ntime)

                for tt,t in enumerate(time):
                    month[tt] = t.month
            except:
                print(model_in, site_name, 'not exists')
                continue

            if per_LAI:
                if model_in in models_calc_LAI:
                    # print('in models_calc_LAI', model_in)
                    LAI = read_LAI_model(site_name, model_in, model_LAI_names[model_in])
                else:
                    # print('not in models_calc_LAI', model_in)
                    LAI = read_LAI_obs(site_name, PLUMBER2_met_path)
                var_tmp = var_tmp/LAI
            else:
                var_tmp = var_tmp
            
            # groupby month
            Var_tmp          = pd.DataFrame(var_tmp,columns=[var_name])
            Var_tmp['month'] = month
            var_t            = Var_tmp.groupby(['month']).mean(numeric_only=True)
            
            var[:,i]         = var_t[var_name].values*3600*24*30
                
        with nc.Dataset(file_path[0], mode='r') as f_flux:
            
            lat[i] = f_flux.variables['latitude'][0,0] 
            lon[i] = f_flux.variables['longitude'][0,0] 

    for m in np.arange(1,13,1):
        var_out              = pd.DataFrame(var[m-1,:], columns=[var_name])
        var_out['lat']       = lat
        var_out['lon']       = lon
        var_out['site_name'] = Site_name

        if per_LAI:
            var_out.to_csv(f'./txt/{var_name}_monthly_mean_per_LAI/{var_name}_month{m}_mean_per_LAI_{model_in}.csv', index=False)
        else:
            var_out.to_csv(f'./txt/{var_name}_monthly_mean/{var_name}_month{m}_mean_{model_in}.csv', index=False)

In [20]:
# Define a function to generate each plot
def save_monthly_mean_parallal(var_name, per_LAI=False):
    
    PLUMBER2_path_site = "/srv/ccrc/LandAP/z5218916/script/PLUMBER2/LSM_GPP_PLUMBER2/nc_files/AU-How.nc"
    f                  = nc.Dataset(PLUMBER2_path_site, mode='r')
    model_list         = f.variables[f'{var_name}_models'][:]
    model_list         = model_list.tolist()
    model_list.append('obs')
    f.close()

    # Create a pool of workers (28 CPUs)
    with mp.Pool(processes=28) as pool:
        # Distribute the tasks across CPUs
        pool.starmap(save_monthly_mean, [(var_name, model_in, per_LAI) for model_in in model_list])

In [21]:
var_name = 'NEE'
per_LAI  = False
save_monthly_mean_parallal(var_name, per_LAI=per_LAI)

In [ ]:
var_name = 'NEE'
per_LAI  = True
save_monthly_mean_parallal(var_name, per_LAI=per_LAI)

/jobfs/125063793.gadi-pbs/ipykernel_248130/2944749280.py:46: RuntimeWarning: divide by zero encountered in divide
  var_tmp = var_tmp/LAI
/jobfs/125063793.gadi-pbs/ipykernel_248130/2944749280.py:46: RuntimeWarning: divide by zero encountered in divide
  var_tmp = var_tmp/LAI
/jobfs/125063793.gadi-pbs/ipykernel_248130/2944749280.py:46: RuntimeWarning: divide by zero encountered in divide
  var_tmp = var_tmp/LAI
/jobfs/125063793.gadi-pbs/ipykernel_248130/2944749280.py:46: RuntimeWarning: divide by zero encountered in divide
  var_tmp = var_tmp/LAI
/jobfs/125063793.gadi-pbs/ipykernel_248130/2944749280.py:46: RuntimeWarning: divide by zero encountered in divide
  var_tmp = var_tmp/LAI
/jobfs/125063793.gadi-pbs/ipykernel_248130/2944749280.py:46: RuntimeWarning: divide by zero encountered in divide
  var_tmp = var_tmp/LAI
/jobfs/125063793.gadi-pbs/ipykernel_248130/2944749280.py:46: RuntimeWarning: divide by zero encountered in divide
  var_tmp = var_tmp/LAI
/jobfs/125063793.gadi-pbs/ipykerne

In [ ]:
var_name = 'GPP'
per_LAI  = False
save_monthly_mean_parallal(var_name, per_LAI=per_LAI)

In [ ]:
var_name = 'GPP'
per_LAI  = True
save_monthly_mean_parallal(var_name, per_LAI=per_LAI)